In [1]:
library(poppr)
library(vcfR)
library(adegenet)
library(parallel)

Loading required package: adegenet

Loading required package: ade4


   /// adegenet 2.1.11 is loaded ////////////

   > overview: '?adegenet'
   > tutorials/doc/questions: 'adegenetWeb()' 
   > bug reports/feature requests: adegenetIssues()



Registered S3 method overwritten by 'pegas':
  method      from
  print.amova ade4

This is poppr version 2.9.8. To get started, type package?poppr
OMP parallel support: available


   *****       ***   vcfR   ***       *****
   This is vcfR 1.16.0 
     browseVignettes('vcfR') # Documentation
     citation('vcfR') # Citation
   *****       *****      *****       *****




In [2]:
metafile <- '../../../../hybrid_diversity/5_snp_analyses/6_hybrid_analyses/metadata.txt'
vcffile <- '../../../../hybrid_diversity/5_snp_analyses/4_chloroplast/chloro_variants_qual30_maf5_nomissing_nohets_justhybrids.recode.vcf'

In [3]:
cat("Loading VCF file...\n")
# Read the VCF - adjust path as needed
vcf <- read.vcfR(vcffile)
 
# Convert to genind object (required for AMOVA)
cat("Converting to genind object...\n")
gen <- vcfR2genind(vcf)
 
cat("Sample names in VCF:\n")
print(indNames(gen))

Loading VCF file...
Scanning file to determine attributes.
File attributes:
  meta lines: 68
  header_line: 69
  variant count: 188
  column count: 31
Meta line 68 read in.
All meta lines processed.
gt matrix initialized.
Character matrix gt created.
  Character matrix gt rows: 188
  Character matrix gt cols: 31
  skip: 0
  nrows: 188
  row_num: 0
Processed variant: 188
All variants processed
Converting to genind object...
Sample names in VCF:
 [1] "12A1"         "12A1-SC1"     "12A1-SC2"     "12A1-SC3"     "13A5"        
 [6] "8020"         "8027"         "ARC66"        "ARC81"        "ARC82"       
[11] "ARC83"        "ARC84"        "PK2297D4"     "PK2298C6"     "PK2299B3"    
[16] "PK2300D3"     "UTEX2797"     "UTEX2797-SC1" "UTEX2797-SC2" "UTEX2797-SC3"
[21] "UTEX2797-SC4" "UTEX2827"    


In [4]:
cat("\nLoading metadata...\n")
metadata <- read.table(metafile, sep = "\t", stringsAsFactors = FALSE, header = TRUE)
print(head(metadata))


Loading metadata...
  strain_ID cytotype    region
1      8020       A1     Texas
2     ARC66       A1 Carolinas
3     ARC81       A1 Carolinas
4     ARC82       A1 Carolinas
5     ARC83       A1 Carolinas
6     ARC84       A1 Carolinas


In [5]:
# Check that all VCF samples have metadata
missing_meta <- setdiff(indNames(gen), metadata$strain_ID)
if (length(missing_meta) > 0) {
  stop("Missing metadata for strains: ", paste(missing_meta, collapse = ", "))
}

In [6]:
# Reorder metadata to match VCF sample order
metadata <- metadata[match(indNames(gen), metadata$strain_ID), ]

In [9]:
# Attach metadata as strata to the genind object
strata_df <- data.frame(
  strain_ID = metadata$strain_ID,
  cytotype = factor(metadata$cytotype),
  region = factor(metadata$region)
)
strata(gen) <- strata_df
 
cat("\nStrata summary:\n")
print(table(strata_df$cytotype, strata_df$region))



Strata summary:
    
     Carolinas Texas
  A1         5     8
  A2         1     8


In [11]:
# Set population structure and run AMOVA without within-individual variance
setPop(gen) <- ~region/cytotype
amova_result1 <- poppr.amova(gen, ~region/cytotype, method = "pegas", within = FALSE, nperm = 999)
cat("\nAMOVA ~region/cytotype\n")
print(amova_result1)


 No missing values detected.


Warning message in is.euclid(xdist):
“Zero distance(s)”



AMOVA ~region/cytotype

	Analysis of Molecular Variance

Call: pegas::amova(formula = hier, data = hierdf, nperm = nperm, is.squared = FALSE)

               SSD        MSD df
region    195.3674 195.367424  1
cytotype 1696.7917 848.395833  2
Error      37.7500   2.097222 18
Total    1929.9091  91.900433 21

Variance components:
           sigma2 P.value
region   -84.8577   0.999
cytotype 175.0963   0.000
Error      2.0972        

Phi-statistics:
  region.in.GLOBAL (Phi_CT) cytotype.in.GLOBAL (Phi_ST) 
                 -0.9190126                   0.9772870 
cytotype.in.region (Phi_SC) 
                  0.9881642 

Variance coefficients:
       a        b        c 
4.833333 5.333333 8.727273 



In [12]:
# Test the other hierarchy
setPop(gen) <- ~cytotype/region  
amova_result2 <- poppr.amova(gen, ~cytotype/region, method = "pegas", within = FALSE, nperm = 999)
cat("\nAMOVA ~cytotype/region\n")
print(amova_result2)


 No missing values detected.


Warning message in is.euclid(xdist):
“Zero distance(s)”



AMOVA ~cytotype/region

	Analysis of Molecular Variance

Call: pegas::amova(formula = hier, data = hierdf, nperm = nperm, is.squared = FALSE)

                SSD         MSD df
cytotype 1859.82362 1859.823621  1
region     32.33547   16.167735  2
Error      37.75000    2.097222 18
Total    1929.90909   91.900433 21

Variance components:
           sigma2 P.value
cytotype 172.3003   0.349
region     3.5480   0.003
Error      2.0972        

Phi-statistics:
cytotype.in.GLOBAL (Phi_CT)   region.in.GLOBAL (Phi_ST) 
                  0.9682758                   0.9882142 
region.in.cytotype (Phi_SC) 
                  0.6284930 

Variance coefficients:
        a         b         c 
 3.965812  7.068376 10.636364 



In [13]:
# Set population structure and run AMOVA without within-individual variance
amova_result3 <- poppr.amova(gen, ~region, method = "pegas", within = FALSE, nperm = 999)
cat("\nAMOVA just region\n")
print(amova_result3)


 No missing values detected.


Warning message in is.euclid(xdist):
“Zero distance(s)”



AMOVA just region

	Analysis of Molecular Variance

Call: pegas::amova(formula = hier, data = hierdf, nperm = nperm, is.squared = FALSE)

             SSD       MSD df
region  195.3674 195.36742  1
Error  1734.5417  86.72708 20
Total  1929.9091  91.90043 21

Variance components:
       sigma2 P.value
region 12.448   0.091
Error  86.727        

Phi-statistics:
region.in.GLOBAL 
       0.1255187 

Variance coefficients:
       a 
8.727273 



In [16]:
# Test the other hierarchy
amova_result4 <- poppr.amova(gen, ~cytotype, method = "pegas", within = FALSE, nperm = 999)
cat("\nAMOVA just cytotype\n")
print(amova_result4)


 No missing values detected.


Warning message in is.euclid(xdist):
“Zero distance(s)”



AMOVA just cytotype

	Analysis of Molecular Variance

Call: pegas::amova(formula = hier, data = hierdf, nperm = nperm, is.squared = FALSE)

                SSD         MSD df
cytotype 1859.82362 1859.823621  1
Error      70.08547    3.504274 20
Total    1929.90909   91.900433 21

Variance components:
           sigma2 P.value
cytotype 174.5258       0
Error      3.5043        

Phi-statistics:
cytotype.in.GLOBAL 
         0.9803164 

Variance coefficients:
       a 
10.63636 

